In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "tifffile",
#     "spotiflow",
#     "tqdm",
#     "napari[all]"
# ]
# ///

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spotiflow.model import Spotiflow, SpotiflowTrainingConfig
from spotiflow.utils import get_data

In [ ]:
data_path = "data/05_spot_detection_spotiflow_finetuning"

train_imgs, train_spots, val_imgs, val_spots, test_imgs, test_spots = get_data(
    data_path, include_test=True
)

In [ ]:
index = 0
img = train_imgs[index]
spots = train_spots[index]

# set the contrast limits to the 1st and 99.8th percentiles of the image pixel values
plt.imshow(img, cmap="gray", clim=tuple(np.percentile(img, (1, 98))))
plt.scatter(spots[:, 1], spots[:, 0], facecolors="none", edgecolors="green")
plt.axis("off")
plt.title("Example Training Image")
plt.show()

In [ ]:
model = Spotiflow.from_pretrained("synth_complex")

In [ ]:
train_config = SpotiflowTrainingConfig(
    num_epochs=30,
    finetuned_from="synth_complex",  # optional, good for keeping track
)

In [ ]:
model.fit(
    train_imgs,
    train_spots,
    val_imgs,
    val_spots,
    save_dir="data/05_spot_detection_spotiflow_finetuning/finetuned_model",
    train_config=train_config,
)

In [ ]:
# load the pretrained model
synth_complex = Spotiflow.from_pretrained("synth_complex")

# run both the pretrained and finetuned models on the test set
img_idx = 3  # index of the test image to evaluate
points_pretrained, _ = synth_complex.predict(test_imgs[img_idx])
points_finetuned, _ = model.predict(test_imgs[img_idx])

In [ ]:
# 2 columns, 1 row
plt.figure(figsize=(10, 5))
clim = tuple(np.percentile(test_imgs[img_idx], (1, 98)))

plt.subplot(1, 2, 1)
plt.imshow(test_imgs[img_idx], cmap="gray", clim=clim)
plt.scatter(
    points_pretrained[:, 1],
    points_pretrained[:, 0],
    facecolors="none",
    edgecolors="magenta",
    alpha=0.7,
)
plt.title("Pretrained")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(test_imgs[img_idx], cmap="gray", clim=clim)
plt.scatter(
    points_finetuned[:, 1],
    points_finetuned[:, 0],
    facecolors="none",
    edgecolors="green",
    alpha=0.7,
)
plt.title("Finetuned")
plt.axis("off")
plt.show()